In [ ]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model # langchain 은 langgraph의 자매 프로젝트 같은 것, 단지 ai 모델이랑 쉽게 대화할 수 잇게 해주는
from langgraph.graph.message import MessagesState
# ToolNode: tools를 호출하는 역할 / tools_condition:  state에서 messages를 꺼내서 거기에 tool call이 있는지 감지하는 역할을 하고 그 조건이 folw(작업흐름)를 tool 노드로 보내줌 
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini") 

conn = sqlite3.connect(
    "memory3.db",
    check_same_thread=False,
)


In [ ]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [ ]:
@tool 
def get_weather(city:str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny"

llm_with_tools = llm.bind_tools(tools=[get_weather]) # langchain임 , tools 존재 알려주기 

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(
    tools=[get_weather],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition) # tools_condition이 "chatbot -> tools"로 보내거나 "chatbot -> end로 보냄"
graph_builder.add_edge("tools","chatbot") # tools_condition이 tools로 보낸다면 tool을 실행하고 output을 기록함 -> chatbot으로 돌아가서 ai에게 "이게 내가 얻은 tool output이야" 라고 알려주고 -> ai 다시 응답 ->  response로 다시 tools_condition돌아감 -> 더이상 tool calling이 없음을 확인 후 end로 

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [ ]:
graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is the weather in machupichu"},
        ]
    },
    config={
        "configurable" : {
            "thread_id" : "2"# 유저의 ID
      },
        "recursion_limit" : 2 # recursion limit(그래프가 몇 step까지 실행될 지 상한):  그래흐가 얼마나 오랫동안 실행될 수 있는지 알려줌, 1이면 노드가 한번 반복후 끝이고 안정장치임(무한루프X)
    }
)

In [ ]:
graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "내가 방금 뭘 물어봣지?"},
        ]
    },
    config={
        "configurable" : {  # 그래프에 설정값을 넘길 수 있는 구조임 
            "thread_id" : "2"# 유저의 ID
        },
       #  "recursion_limit" : 2 # recursion limit(그래프가 몇 step까지 실행될 지 상한):  그래흐가 얼마나 오랫동안 실행될 수 있는지 알려줌, 1이면 노드가 한번 반복후 끝이고 안정장치임(무한루프X)
    }
)

In [ ]:
for state in graph.get_state_history(
    {
        "configurable" : {
            "thread_id" : "2",
        }
    }
):
    print(state.next)   #위에서 아래로 봄 , 메모리만 주는게 아니라 그래프가 뭘 했는지 가시성도 줌 

('chatbot',)
('tools',)
('chatbot',)
('__start__',)
()
('chatbot',)
('__start__',)
